In [7]:
import pandas as pd
import sys
sys.path.append('../mirt-official') 
from load_params import load_and_rotate

resmat = pd.read_pickle("../data/resmat.pkl")

theta, a, b = load_and_rotate("../data/mirt_model_k30_rep0.pt")

--- Loading from Cache ---
Loaded cached theta shape: (183, 30)
Loaded cached a shape: (78712, 30)
Loaded cached b shape: (78712,)



In [32]:
import numpy as np
import torch
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score

# ====================================================
# 1. Parallel Analysis on A (discrimination matrix)
# ====================================================
def parallel_analysis_A(A, M=1000, alpha=0.95, random_state=42):
    """
    Run parallel analysis on discrimination matrix A (J x K0).
    Returns the number of retained factors.
    """
    rng = np.random.RandomState(random_state)
    K0 = A.shape[1]

    # Real eigenvalues
    eig_real = np.linalg.eigvalsh(A.T @ A)[::-1]

    # Null eigenvalues (column permutations)
    nulls = np.zeros((M, K0))
    for m in range(M):
        A_perm = np.column_stack([rng.permutation(A[:, k]) for k in range(K0)])
        nulls[m] = np.linalg.eigvalsh(A_perm.T @ A_perm)[::-1]

    thresh = np.percentile(nulls, alpha*100, axis=0)
    K_star = np.sum(eig_real > thresh)
    return K_star, eig_real, thresh


# ====================================================
# 2. Cross-validation predictive check
# ====================================================
def crossval_auc_single(theta, a, b, Y_obs, pairs, K, n_splits=5, device="cpu"):
    N, K0 = theta.shape
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    aucs = []

    for train_idx, test_idx in kf.split(pairs):
        test_pairs = pairs[test_idx]
        test_y = Y_obs[test_idx]

        # Restrict to top-K factors
        theta_K = theta[:, :K]
        a_K = a[:, :K]

        # Predict
        with torch.no_grad():
            logits = (theta_K[test_pairs[:,0]] * a_K[test_pairs[:,1]]).sum(axis=1) - b[test_pairs[:,1]]
            probs = torch.sigmoid(torch.tensor(logits)).numpy()

        aucs.append(roc_auc_score(test_y, probs))

    return np.mean(aucs), np.std(aucs)
    
def crossval_auc(theta, a, b, Y_obs, pairs, K_list, n_splits=5, device="cpu"):
    """
    Cross-validated predictive check for candidate K values.
    theta: (N x K0)
    a: (J x K0)
    b: (J,)
    Y_obs: observed responses (len pairs,)
    pairs: array of (i, j) indices
    """
    results = {}
    N, K0 = theta.shape
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    for K in K_list:
        aucs = []
        for train_idx, test_idx in kf.split(pairs):
            train_pairs, test_pairs = pairs[train_idx], pairs[test_idx]
            test_y = Y_obs[test_idx]

            # Restrict to top-K factors
            theta_K = theta[:, :K]
            a_K = a[:, :K]

            # Predict
            with torch.no_grad():
                logits = (theta_K[test_pairs[:,0]] * a_K[test_pairs[:,1]]).sum(axis=1) - b[test_pairs[:,1]]
                probs = torch.sigmoid(torch.tensor(logits)).numpy()

            aucs.append(roc_auc_score(test_y, probs))

        results[K] = (np.mean(aucs), np.std(aucs))
    return results


# ====================================================
# 3. Residual diagnostics (Yen’s Q3)
# ====================================================
def compute_residuals(theta, a, b, pairs, Y_obs, K):
    """
    Compute residuals r_ij = Y_ij - p_ij for given K.
    """
    theta_K = theta[:, :K]
    a_K = a[:, :K]

    logits = (theta_K[pairs[:,0]] * a_K[pairs[:,1]]).sum(axis=1) - b[pairs[:,1]]
    probs = 1 / (1 + np.exp(-logits))
    residuals = Y_obs - probs
    return residuals


def q3_statistic(residuals, pairs, n_persons, n_items, max_pairs=10000, random_state=42):
    """
    Compute Yen's Q3 by correlating residuals between item pairs.
    Subsamples item pairs for speed.
    """
    rng = np.random.RandomState(random_state)

    # Build sparse residual matrix (items x persons)
    R = {j: {} for j in range(n_items)}
    for (i, j), r in zip(pairs, residuals):
        R[j][i] = r

    items = list(R.keys())
    n_total_pairs = len(items) * (len(items) - 1) // 2
    sample_pairs = min(max_pairs, n_total_pairs)

    q3_vals = []
    for _ in range(sample_pairs):
        j1, j2 = rng.choice(items, size=2, replace=False)
        common = set(R[j1].keys()) & set(R[j2].keys())
        if len(common) > 2:  # at least 3 shared respondents
            r1 = np.array([R[j1][i] for i in common])
            r2 = np.array([R[j2][i] for i in common])
            corr = np.corrcoef(r1, r2)[0, 1]
            if not np.isnan(corr):
                q3_vals.append(corr)

    return np.mean(q3_vals), np.percentile(q3_vals, [5, 95])



# ====================================================
# 4. Interpretability check (factor strength)
# ====================================================
def interpretability_check(a, K, threshold=0.1, min_items=10):
    """
    Check each factor for number of items with |loading| >= threshold.
    """
    strong_counts = []
    for d in range(K):
        count = np.sum(np.abs(a[:,d]) >= threshold)
        strong_counts.append(count)
    interpretable = [c >= min_items for c in strong_counts]
    return strong_counts, interpretable

from sklearn.metrics import mean_squared_error

def compute_fit_indices(theta, a, b, Y_obs, pairs, k, n_persons, n_items):
    """
    Compute global model fit indices for MIRT.
    theta: (N x K)
    a: (J x K)
    b: (J,)
    Y_obs: observed binary responses
    pairs: (N_obs x 2) indices (i, j)
    k: number of factors
    n_persons, n_items: matrix size
    """

    # --- Predicted probabilities ---
    logits = (theta[pairs[:,0], :k] * a[pairs[:,1], :k]).sum(axis=1) - b[pairs[:,1]]
    probs = 1 / (1 + np.exp(-logits))

    # --- Log-likelihood ---
    ll = np.sum(Y_obs * np.log(probs + 1e-9) + (1 - Y_obs) * np.log(1 - probs + 1e-9))
    neg2ll = -2 * ll

    # --- Chi-square statistic (Pearson) ---
    # Expected counts vs observed counts
    expected = probs
    residuals = (Y_obs - expected) / np.sqrt(expected * (1 - expected) + 1e-9)
    chi_square = np.sum(residuals**2)

    # --- Degrees of freedom ---
    # Rough df = (#observations - #parameters)
    n_obs = len(Y_obs)
    n_params = n_persons * k + n_items * k + n_items
    df = max(n_obs - n_params, 1)  # avoid negative

    # --- RMSEA ---
    rmsea = np.sqrt(max((chi_square - df), 0) / (df * n_obs))

    # --- SRMR ---
    # Standardized root mean square residual between obs & expected
    srmr = np.sqrt(mean_squared_error(Y_obs, expected))

    return {
        "-2LL": neg2ll,
        "Chi2": chi_square,
        "df": df,
        "RMSEA": rmsea,
        "SRMR": srmr
    }



In [2]:
import numpy as np

def parallel_analysis_A_return_nulls(A, M=1000, alpha=0.95, random_state=42):
    """
    Parallel analysis on discrimination matrix A (J x K0).
    Returns:
      - K_star: number of retained factors (using alpha quantile)
      - eig_real: real eigenvalues (K0,) descending
      - thresh: alpha-quantile thresholds (K0,) descending
      - nulls: array (M, K0) of null eigenvalues (each row descending)
    """
    rng = np.random.RandomState(random_state)
    K0 = A.shape[1]

    eig_real = np.linalg.eigvalsh(A.T @ A)[::-1]   # descending

    nulls = np.zeros((M, K0))
    for m in range(M):
        # permute rows independently for each column (preserves column marginals)
        A_perm = np.column_stack([rng.permutation(A[:, k]) for k in range(K0)])
        nulls[m] = np.linalg.eigvalsh(A_perm.T @ A_perm)[::-1]

    thresh = np.percentile(nulls, alpha*100, axis=0)
    K_star = np.sum(eig_real > thresh)
    return K_star, eig_real, thresh, nulls

# --- Run PA and get nulls ---
K_star, eig_real, thresh, nulls = parallel_analysis_A_return_nulls(a, M=1000, alpha=0.95, random_state=0)

# --- Compute PA p-values per component ---
def pa_pvalues(eig_real, nulls):
    # nulls: (M, K0), eig_real: (K0,)
    # p_k = proportion of null eigenvalues >= observed
    pvals = np.mean(nulls >= eig_real[None, :], axis=0)
    return pvals

pvals = pa_pvalues(eig_real, nulls)

# --- Pretty print first 20 components (or all) ---
for k in range(min(len(eig_real), 20)):
    print(f"k={k+1:2d}  eig={eig_real[k]: .6e}  thresh95={thresh[k]: .6e}  p={pvals[k]:.4f}")
print("Parallel Analysis suggests K* =", K_star)


k= 1  eig= 3.353024e+04  thresh95= 3.136592e+04  p=0.0000
k= 2  eig= 3.129951e+04  thresh95= 2.859559e+04  p=0.0000
k= 3  eig= 2.588004e+04  thresh95= 2.481289e+04  p=0.0000
k= 4  eig= 1.915819e+04  thresh95= 1.988048e+04  p=1.0000
k= 5  eig= 1.505569e+04  thresh95= 1.773143e+04  p=1.0000
k= 6  eig= 1.399469e+04  thresh95= 1.389254e+04  p=0.0000
k= 7  eig= 1.275671e+04  thresh95= 1.348555e+04  p=1.0000
k= 8  eig= 1.254068e+04  thresh95= 1.221536e+04  p=0.0000
k= 9  eig= 1.180520e+04  thresh95= 1.211727e+04  p=1.0000
k=10  eig= 1.162056e+04  thresh95= 1.174339e+04  p=1.0000
k=11  eig= 1.144320e+04  thresh95= 1.126386e+04  p=0.0000
k=12  eig= 1.140486e+04  thresh95= 1.097748e+04  p=0.0000
k=13  eig= 1.127973e+04  thresh95= 1.087456e+04  p=0.0000
k=14  eig= 1.097701e+04  thresh95= 1.079620e+04  p=0.0000
k=15  eig= 1.061936e+04  thresh95= 1.072991e+04  p=0.9980
k=16  eig= 1.057565e+04  thresh95= 1.065315e+04  p=0.9240
k=17  eig= 1.042919e+04  thresh95= 1.051417e+04  p=0.9870
k=18  eig= 1.0

In [34]:
# Load response matrix indices (pairs) and observed values
pairs = np.argwhere(~resmat.isna().values)
Y_obs = resmat.values[pairs[:,0], pairs[:,1]].astype(float)

import numpy as np
import torch
import pandas as pd
from tqdm import tqdm

# Define candidate Ks and repetitions
K_list = [1, 2, 3, 4, 5, 6, 7, 8]
reps = range(1)  # adjust if you trained more/less reps

# Store per-run results in a dataframe
all_results = []

# Progress bar over all (K, rep) combinations
for k in K_list:
    for r in tqdm(reps, desc=f"Evaluating K={k}", leave=False):
        model_path = f"../data/mirt_model_k{k}_rep{r}.pt"

        try:
            state = torch.load(model_path, map_location="cpu", weights_only=False)
        except FileNotFoundError:
            print(f"⚠️ Skipping missing file: {model_path}")
            continue

        theta, a, b = state['theta'].numpy(), state['a'].numpy(), state['b'].numpy()

        # --- 1. Cross-validation predictive check ---
        auc_dict = crossval_auc(theta, a, b, Y_obs, pairs, [k])
        auc_mean, auc_se = auc_dict[k]

        # --- 2. Residual diagnostics (Q3) ---
        residuals = compute_residuals(theta, a, b, pairs, Y_obs, k)
        q3_mean, q3_range = q3_statistic(residuals, pairs, n_persons=theta.shape[0], n_items=a.shape[0])

        # --- 3. Interpretability check ---
        counts, interp = interpretability_check(a, k)

        # --- 4. Fit indices ---
        fit_stats = compute_fit_indices(theta, a, b, Y_obs, pairs, k,
                                n_persons=theta.shape[0], n_items=a.shape[0])

        # Collect results
        all_results.append({
            "K": k,
            "Rep": r,
            "CV_AUC_mean": auc_mean,
            "CV_AUC_se": auc_se,
            "Q3_mean": q3_mean,
            "Q3_low": q3_range[0],
            "Q3_high": q3_range[1],
            "Items_per_factor": counts,
            "Interpretable": bool(all(interp)),
            "-2LL": fit_stats["-2LL"],
            "Chi2": fit_stats["Chi2"],
            "df": fit_stats["df"],
            "RMSEA": fit_stats["RMSEA"],
            "SRMR": fit_stats["SRMR"]
        })

# Convert to DataFrame
results_df = pd.DataFrame(all_results)

# Save to CSV
results_df.to_csv("../data/mirt_evaluation_results.csv", index=False)
print("\n✅ Results saved to ../data/mirt_evaluation_results.csv")


✅ Results saved to ../data/mirt_evaluation_results.csv


In [17]:
# --- Aggregated summaries ---
print("\n=== Aggregated CV AUC Results ===")
for k in K_list:
    subset = results_df[results_df["K"] == k]
    print(f"K={k}: mean={subset['CV_AUC_mean'].mean():.4f}, "
          f"std={subset['CV_AUC_mean'].std():.4f}, n={len(subset)}")

print("\n=== Aggregated Q3 Results ===")
for k in K_list:
    subset = results_df[results_df["K"] == k]
    print(f"K={k}: Q3 mean={subset['Q3_mean'].mean():.3f}, "
          f"std={subset['Q3_mean'].std():.3f}")

print("\n=== Interpretability Summary ===")
for k in K_list:
    subset = results_df[results_df["K"] == k]
    n_interp = subset["Interpretable"].sum()
    print(f"K={k}: Interpretable runs={n_interp} / {len(subset)}")



=== Aggregated CV AUC Results ===
K=9: mean=0.9318, std=0.0018, n=5
K=10: mean=0.9343, std=0.0022, n=5
K=11: mean=0.9344, std=0.0011, n=5

=== Aggregated Q3 Results ===
K=9: Q3 mean=0.009, std=0.002
K=10: Q3 mean=0.009, std=0.003
K=11: Q3 mean=0.009, std=0.001

=== Interpretability Summary ===
K=9: Interpretable runs=5 / 5
K=10: Interpretable runs=5 / 5
K=11: Interpretable runs=5 / 5


# Interpretability check

In [94]:
from collections import Counter, defaultdict
import numpy as np

def interpretability_check_weighted(a, k, item_domains, threshold=0.1, topn=3):
    """
    Interpretability check with inverse weighting by domain size.
    Each domain contributes proportionally, regardless of item count.
    """
    J, K0 = a.shape
    # Precompute domain sizes
    domain_sizes = Counter(item_domains)

    summary = []
    for f in range(k):
        loadings = a[:, f]
        strong_idx = np.where(np.abs(loadings) >= threshold)[0]

        # Weighted domain counts
        weighted_counts = defaultdict(float)
        for j in strong_idx:
            dom = item_domains[j]
            weighted_counts[dom] += 1.0 / domain_sizes[dom]  # inverse weighting

        # Normalize to sum=1
        total_weight = sum(weighted_counts.values())
        normed_counts = {dom: val / total_weight for dom, val in weighted_counts.items()}

        # Pick top domains
        top_domains = sorted(normed_counts.items(), key=lambda x: x[1], reverse=True)[:topn]

        summary.append({
            "Factor": f + 1,
            "Num_items": len(strong_idx),
            "Top_domains_weighted": top_domains
        })

    return summary


In [95]:
import sys
sys.path.append('.')
import pandas as pd
from load_rotate_on_pc import load_and_rotate_pc1_reckase

resmat = pd.read_pickle("../data/resmat.pkl")
model_names = resmat.index
factor_names = [f'F{i+1}' for i in range(theta.shape[1])]
ability_df = pd.DataFrame(theta, index=model_names, columns=factor_names)
# Extract names for labeling
item_names = resmat.columns.get_level_values('input.text').to_list()
model_names = resmat.index.to_list()

df_theta = pd.DataFrame(theta, index=model_names, columns=factor_names)
df_a = pd.DataFrame(a, index=resmat.columns.get_level_values('scenario'), columns=factor_names)

df_theta, df_a, top_items_df, Q_final = load_and_rotate_pc1_reckase(
    model_path='../data/mirt_model_k2_rep0.pt',
    top_k=20,
    model_names=model_names,
    item_names=item_names
)

/Users/ronan/Developer/reeval-multi/mirt-official/load_rotate_on_pc.py:212: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_data = torch.load(model_path, map_location=to

In [97]:
# _, a_k, __ = load_and_rotate("../data/mirt_model_k9_rep0.pt")
a_k = df_a.values
resmat = pd.read_pickle("../data/resmat.pkl")

item_domains = resmat.columns.get_level_values('scenario').tolist()

summary = interpretability_check_weighted(a_k, k=2, item_domains=item_domains, threshold=0.1, topn=10)

for s in summary:
    print(f"Factor {s['Factor']}: {s['Num_items']} items")
    print("   Top domains:", s['Top_domains_weighted'])


Factor 1: 64816 items
   Top domains: [('civil_comments', 0.06614187108675554), ('gsm', 0.06515562028034906), ('commonsense', 0.06508820449937991), ('math', 0.06341986988114999), ('mmlu', 0.06195359316884498), ('med_qa', 0.06117961136754523), ('thai_exam', 0.05855005107339072), ('legalbench', 0.055617080198377675), ('air_bench_2024', 0.055441992165234696), ('entity_data_imputation', 0.04404475585689354)]
Factor 2: 68143 items
   Top domains: [('entity_data_imputation', 0.05167904805762761), ('imdb', 0.05163631624609381), ('entity_matching', 0.049150489044743256), ('air_bench_2024', 0.048967639648402436), ('boolq', 0.048957753980091744), ('raft', 0.04775799729484471), ('synthetic_reasoning', 0.04714674271260405), ('commonsense', 0.0465702559374626), ('civil_comments', 0.04645070896565684), ('wikifact', 0.046339874944840743)]


# Confirmation of MIRT dimension

In [91]:
training_result = pd.read_csv('../data/mirt_comparison_repeated.csv')
# eval_result = pd.read_csv('../data/mirt_evaluation_results.csv').rename(columns={'Rep': 'Repetition'})

# join two datasets on K and Repetition
# training_result = training_result.merge(eval_result, on=['K', 'Repetition'], how='inner')
training_result.sort_values(by=['K', 'Repetition'], ascending=True)

,Repetition,K,Test AUC,AIC,BIC,Num Params,LogLikelihood
23,0,1,0.869515,3901898.50,5.992810e+06,157607,-1793342.250
24,1,1,0.871917,3867441.00,5.958352e+06,157607,-1776113.500
27,2,1,0.872367,3867039.75,5.957951e+06,157607,-1775912.875
30,3,1,0.872097,3874511.25,5.965422e+06,157607,-1779648.625
33,4,1,0.869780,3892708.50,5.983620e+06,157607,-1788747.250
18,0,2,0.885265,3723838.25,6.861419e+06,236502,-1625417.125
25,1,2,0.884595,3719548.25,6.857129e+06,236502,-1623272.125
28,2,2,0.883583,3758772.50,6.896353e+06,236502,-1642884.250
31,3,2,0.883879,3733345.00,6.870926e+06,236502,-1630170.500
34,4,2,0.881244,3783133.50,6.920714e+06,236502,-1655064.750


In [92]:
# Drop non number columns, group by K and Repetition and take the mean of the rest
training_result = training_result.select_dtypes(include=[np.number])
# Group by K and Repetition and average those groups to get a single value for each K
final_result = training_result.copy().drop(columns=['Repetition']).groupby(['K']).mean().reset_index()
final_result.to_csv('../output/mirt_comparison_merge.csv', index=False)
final_result

,K,Test AUC,AIC,BIC,Num Params,LogLikelihood
0,1,0.871135,3880719.80,5.971631e+06,157607.0,-1782752.900
1,2,0.883713,3743727.50,6.881308e+06,236502.0,-1635361.750
2,3,0.888146,3733654.95,7.917905e+06,315397.0,-1551430.475
3,4,0.890847,3764988.70,8.995908e+06,394292.0,-1488202.350
4,5,0.891193,3817704.70,1.009529e+07,473187.0,-1435665.350
5,6,0.892420,3881868.00,1.120613e+07,552082.0,-1388852.000
6,7,0.893155,3936700.50,1.230763e+07,630977.0,-1337373.250
7,8,0.891814,4123636.00,1.354123e+07,709872.0,-1351946.000
8,9,0.892914,4226226.35,1.469049e+07,788767.0,-1324346.175
9,10,0.892878,4321278.00,1.583221e+07,867662.0,-1292977.000


In [93]:
# Calculate the difference between each row and the one before it
changes_df = final_result.diff()

# The first row is now NaN. Fill it with the original first row's values.
changes_df.iloc[0] = final_result.iloc[0]

# Set the 'K' column to be integers for clarity, as .diff() makes it a float
changes_df['K'] = changes_df['K'].astype(int)

print("Changes in metrics as K increases:")
changes_df

Changes in metrics as K increases:


,K,Test AUC,AIC,BIC,Num Params,LogLikelihood
0,1,0.871135,3880719.80,5.971631e+06,157607.0,-1782752.900
1,1,0.012578,-136992.30,9.096772e+05,78895.0,147391.150
2,1,0.004433,-10072.55,1.036597e+06,78895.0,83931.275
3,1,0.002701,31333.75,1.078003e+06,78895.0,63228.125
4,1,0.000346,52716.00,1.099385e+06,78895.0,52537.000
5,1,0.001227,64163.30,1.110833e+06,78895.0,46813.350
6,1,0.000736,54832.50,1.101502e+06,78895.0,51478.750
7,1,-0.001342,186935.50,1.233605e+06,78895.0,-14572.750
8,1,0.001100,102590.35,1.149260e+06,78895.0,27599.825
9,1,-0.000036,95051.65,1.141721e+06,78895.0,31369.175
